# AlphaMissense Fetch

This notebook demonstrates `run_alphamissense_fetch`, which retrieves per-residue, per-substitution AlphaMissense pathogenicity scores for human proteins from the AlphaFold Protein Structure Database (AlphaFold DB) by UniProt accession. AlphaMissense is a deep learning model that predicts the pathogenicity of all possible single amino acid substitutions in human proteins, distinguishing likely benign from likely pathogenic missense variants. See the [Cheng et al. 2023 paper](https://doi.org/10.1126/science.adg7492) and the [AlphaMissense GitHub](https://github.com/google-deepmind/alphamissense) for details on the model and its training.

In [1]:
from proto_tools.tools.database_retrieval import (
    AlphaMissenseFetchConfig,
    AlphaMissenseFetchInput,
    run_alphamissense_fetch,
)

## Input API Reference

| Field | Type | Required | Description |
|-------|------|----------|-------------|
| `uniprot_id` | `str` | yes | UniProt accession (human; e.g. 'P04637') |

## Config API Reference

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `coordinate_system` | `Literal["uniprot", "hg19", "hg38"]` | `"uniprot"` | Which AFDB CSV to fetch. UniProt mode returns the full saturation grid in protein coordinates (~7,500 rows for TP53); hg19/hg38 modes return only SNV-accessible substitutions in genomic coordinates and additionally populate `chrom`, `pos`, `ref`, `alt`, `transcript_id` on each prediction. |

The wrapper exposes no filter knobs because the AFDB CSV is a static file with no server-side filtering. Filter the returned `predictions` list client-side instead — see the examples below.

## Output API Reference

| Field | Type | Description |
|-------|------|-------------|
| `uniprot_accession` | `str` | UniProt accession looked up |
| `predictions` | `list[AlphaMissensePrediction]` | Per-substitution pathogenicity predictions |
| `num_predictions` | `int` | Number of predictions in the source CSV |
| `mean_pathogenicity` | `float \| None` | Mean pathogenicity score across all predictions |
| `source_url` | `str` | URL of the AlphaMissense CSV fetched |

### `AlphaMissensePrediction` fields

| Field | Type | Mode | Description |
|-------|------|------|-------------|
| `position` | `int` | both | 1-indexed residue position in the canonical UniProt sequence |
| `wild_type_aa` | `str` | both | Single-letter wild-type amino acid at this position |
| `alt_aa` | `str` | both | Single-letter alternate amino acid being scored |
| `pathogenicity_score` | `float` | both | AlphaMissense pathogenicity score (0.0-1.0) |
| `classification` | `AlphaMissenseClass` | both | Class label: 'likely_benign', 'ambiguous', or 'likely_pathogenic' |
| `chrom` | `str \| None` | hg19/hg38 | Chromosome (e.g. `"chr17"`); `None` in UniProt mode |
| `pos` | `int \| None` | hg19/hg38 | 1-indexed genomic position; `None` in UniProt mode |
| `ref` | `str \| None` | hg19/hg38 | Reference allele; `None` in UniProt mode |
| `alt` | `str \| None` | hg19/hg38 | Alternate allele; `None` in UniProt mode |
| `transcript_id` | `str \| None` | hg19/hg38 | GENCODE transcript ID; `None` in UniProt mode |

## Basic Usage

Fetch all AlphaMissense predictions for human TP53 (UniProt accession `P04637`). The full saturation-mutagenesis grid covers every residue in the canonical sequence times the 19 non-self alternate amino acids.

In [ ]:
inputs = AlphaMissenseFetchInput(uniprot_id="P04637")
config = AlphaMissenseFetchConfig()

output = run_alphamissense_fetch(inputs, config)

print(f"UniProt accession: {output.uniprot_accession}")
print(f"Predictions in CSV: {output.num_predictions}")
if output.mean_pathogenicity is not None:
    print(f"Mean pathogenicity: {round(output.mean_pathogenicity, 4)}")

print("\nFirst 5 predictions:")
for p in output.predictions[:5]:
    print(f"  {p.wild_type_aa}{p.position}{p.alt_aa}: {round(p.pathogenicity_score, 4)} ({p.classification})")

## Pathogenic Variants Only

Filter the saturation grid to high-confidence pathogenic variants client-side after the fetch. The wrapper itself returns every row in the CSV; you slice it.

In [ ]:
pathogenic = [p for p in output.predictions if p.pathogenicity_score >= 0.7]

print(f"Total predictions: {output.num_predictions}")
print(f"Pathogenic hits (score >= 0.7): {len(pathogenic)}")

print("\nFirst 5 high-pathogenicity hits:")
for p in pathogenic[:5]:
    print(f"  {p.wild_type_aa}{p.position}{p.alt_aa}: {round(p.pathogenicity_score, 4)} ({p.classification})")

## Specific Positions

Restrict the result to specific 1-indexed residue positions. Below we query the three classic TP53 hotspot residues: R175, R248, and R273. Each position returns 19 alternate amino acid substitutions.

In [ ]:
hotspots = {175, 248, 273}
hotspot_predictions = [p for p in output.predictions if p.position in hotspots]

print(f"Returned predictions: {len(hotspot_predictions)}")

by_position: dict[int, list] = {}
for p in hotspot_predictions:
    by_position.setdefault(p.position, []).append(p)

for position in sorted(by_position):
    preds = by_position[position]
    wild_type = preds[0].wild_type_aa
    print(f"\nPosition {position} (wild-type {wild_type}):")
    for p in preds:
        print(f"  {p.wild_type_aa}{p.position}{p.alt_aa}: {round(p.pathogenicity_score, 4)} ({p.classification})")

## Class-Based Filter

Restrict the output to a specific AlphaMissense class label by filtering `output.predictions` client-side. Below we keep only `likely_benign` substitutions.

In [ ]:
benign = [p for p in output.predictions if p.classification == "likely_benign"]

print(f"Likely-benign predictions: {len(benign)}")

print("\nSample predictions:")
for p in benign[:3]:
    print(f"  {p.wild_type_aa}{p.position}{p.alt_aa}: {round(p.pathogenicity_score, 4)} ({p.classification})")

## Export Results

Fetched results can be saved to JSON format for downstream use in other tools or analysis pipelines.

In [ ]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

# Export the full saturation grid for downstream use.
output.export("alphamissense_predictions", export_path=output_dir, file_format="json")
print(f"Exported to {output_dir / 'alphamissense_predictions.json'}")